
# Introduction

---
Up to this point, we’ve covered the foundational concepts of machine learning. You may have already experimented with various hyperparameters to optimize model performance. In this notebook, you’ll be introduced to several advanced techniques designed to further enhance your models.


This notebook consists the following parts:

- [A: Data retrieval ](#01)
- [B: Feature engineering](#02)
- [C: Model evaluation](#03)
- [D: Ensemble learning](#04)
- [E: Pipelines](#05)
- [F: Bring it all together](#06)
- [G: Bonus: ML OPS](#07)


---

### Learning Objectives
By the end of this two weeks you will be able to:
- Understand the fundamental concepts of ensemble learning.
- Use evaluation techniques to assess models performance.
- Enhance model performance by feature engineering. 
- Build pipelines for model development and preprocessing

---

### Instructions
- Ensure you fully understand the requirements and objectives of the assignment.
- Review the notebooks refered in the tasks
- If you need additional context or clarification, please check the provided videos or background literature.
- Work through each part of the assignment methodically, ensuring all tasks are completed.
- Update your repository with your new created work

### Additional Notes:
- Do not add datafiles to your repository. Repositories with datafiles will not be accepted
- Class solutions should be delivered in python files. Not in notebooks
- When AI tools are used, you must provide proper references and explanations for how they were utilized. Failure to do so will be considered as academic fraud
- The bonus assignment are not mandatory
- Use PEP8 

Good luck!

F.Feenstra

---


<a name='01'></a>
## Part A. Data retrieval

The dataset you can use for this notebook is the lung dataset from Maastricht University. It comprises 89 non-small cell lung cancer (NSCLC) patients records who underwent surgical treatment. The study where the data is from explored the relationship between radiomic imaging features and gene expression profiles. The samples were collected through biopsies at MAASTRO Clinic in The Netherlands, and the dataset is publicly available.

The authors of the related paper discovered that a prognostic radiomic signature, which captures intra-tumor heterogeneity, is closely associated with underlying gene expression patterns. Developing a machine learning model to predict histology from the Clinical and Genetic Lung data can improve diagnostic accuracy and treatment personalization. In this notebook we will develop such a prediction model. 

**Availabel Datasets**:
- Lung metadata dataset [1]
- Gene expression dataset [2]

**Important**
<span style="background-color: lightgreen;color: black">It is also allowed to use your own dataset from your own project if this data is highly dimensional and contains genetic information.</span>

[1] [NSCLC-Radiomics-Genomics](https://wiki.cancerimagingarchive.net/display/Public/NSCLC-Radiomics-Genomics#16056856db10d39adf704eefa53e41edcf5ef41c)

[2] [Gene Expression Data - GSE58661](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE58nnn/GSE58661/matrix/)

[3] Aerts HJWL, Rios Velazquez E, Leijenaar RTH, Parmar C, Grossmann P, Carvalho S, Bussink J, Monshouwer R, Haibe-Kains B, Rietveld D, Hoebers F, Rietbergen MM, Leemans CR, Dekker A, Quackenbush J, Gillies RJ, & Lambin P. (2015). Data From NSCLC-Radiomics-Genomics. The Cancer Imaging Archive. https://doi.org/10.7937/K9/TCIA.2015.L4FRET6Z


### Data retrieval task
- Retrieve the data. (No cleaning needed yet)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

from notebooks.optimization.data_processing import DataProcessor

metadata = pd.read_excel("../../data/Lung3.metadata.xls")
data = pd.read_csv("../../data/GSE58661_series_matrix.txt.gz",
                   compression="gzip", sep="\t",
                   comment="!",index_col=0).transpose()

In [ ]:
#data["title"] = [f"lung_{i}" for i in range(1, data.shape[0]+1)]
data.head()

In [ ]:
print(data.shape)
metadata.head()

---
<a name='02'></a>
## Part B. Feature engineering

Mind you that choosing an algorithm and hyperparameter tuning might not be enough. If your data is of low quality, the algorithm will have poor performance as well. This is where feature engineering comes into play. Feature engineering involves transforming raw data into meaningful features that better represent the underlying problem to the predictive models, ultimately enhancing the model's performance. By selecting, creating, and refining features, you ensure that your data highlights the most relevant patterns and relationships, allowing the algorithm to learn more effectively. Proper feature engineering can often make the difference between a moderate model and a highly accurate one, even more so than the choice of algorithm itself. 
Possible modifications:
- creation of new features derived from original features
- selection of features
- encoding features
- log transformation 
- scaling
- dimension reduction (*e.g.* PCA)

### <span style="background-color: lightyellow;">Feature engineering Task</span>
- Review the [Study Case Feature Engineering notebook](../Study_Cases/study_case_feature_engineering.ipynb).
- Review the [Study Case RNA-seq Preparation notebook](../Study_Cases/study_case_scanpy_object.ipynb).
- Assess which data preparation and feature engineering steps could be beneficial for your dataset.
- Implement a `DataProcessor` class tailored to your dataset and test it in the cell below. **Make sure it's an appropriate sklearn object, such as a `Transformer` or `Pipeline`.**
- Update your repository with a new directory named `optimization`, including the following:
    - The `DataProcessor` class as a Python file, complete with thorough documentation.
    - An evaluation document that details and justifies your choices using a well-reasoned, argumentative approach.



## Feature selection metadata

In [ ]:
import data_processing
summary_statistic, potential_columns = data_processing.DataProcessor.summary_categorial_data(metadata)
print(potential_columns)

Potential columns to be deleted, based on whether the categories inside are too sparse (every row has a different category), or have no variance (every row has the same category label). We still have the label to map to the genetic information here (the title column), and the category we want use as the label to predict (characteristics.tag.histology). We keep the title, and predictor.

In [ ]:
summary_statistic

Additionally, above you can see that the stage.mets is mostly one stage, just like stage.nodes, they don't add much information so we remove them. For sounder biological reasoning, one would have to delve into the biology of these tumours more and that time is not available currently therefor we just base it on the numbers. This leaves us with three metadata columns, the gender and the tumor stage, and its location.

The diameter and tag grade have too many missing values, so we drop these columns as well.

In [ ]:
mask = DataProcessor.remove_low_count_data(data, metadata["characteristics.tag.histology"])

data = data[mask]
metadata = metadata[mask]

y = metadata["characteristics.tag.histology"]
metadata = metadata[["title", "characteristics.tag.gender", "characteristics.tag.stage.primary.tumor", "source.location"]]

We might also want to consider removing the histology that only shows up once as these are too few samples to actually train on. Therefor, we simply remove it.

## Feature selection genetic data

For the genetic data, we're looking for genes that have a nice amount of variance and maybe because we have so many genes it would also be useful to see about co-expression. Co-expressed genes would give us the same signal and whilst it is biologically interesting, it might lead the model to more keenly look at that signal.

Additionally, we know the data is already normalized so we do not have to worry about that.

First we remove low variance genes before scaling. Once we do standard scaling, variance will be equal throughout. This means setting a threshold, and to set a good threshold we want to know what the variance looks like over the genes. We find the variance per column and then look at a histogram of the number genes per amount of variance.

In [ ]:
import matplotlib.pyplot as plt

gene_variance = data.var(axis=0)

print(gene_variance.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50]))

gene_variance.hist(bins=100)

plt.xlabel("Gene variance")
plt.ylabel("Number of genes")
plt.title("Distribution of gene expression variance");
plt.show()

We can see that variance has a pretty big range, but around the 0.2 marks there's a sharp drop. That means most our genes have quite low variance. Low variance genes could still be useful in distinguishing which genes can help find tumours. Therefor, a threshold of around 0.1 is chosen. We remove some genes, but in a sparse manner.

In [ ]:
from sklearn.feature_selection import VarianceThreshold
print(data.shape)
sel = VarianceThreshold(threshold=0.1)
threshold_data = sel.fit_transform(data)
print(threshold_data.shape)

Just by removing low variance genes (a threshold of 0.1), we have removed around 17 thousand genes.

In [ ]:
scaled_data = DataProcessor.scale(threshold_data)

We scale the data so that changes are properly picked up regardless of size, then we removed the low variance genes. Or we try to, as no seem to miss the threshold. From here on, we can merge the scaled data with the metadata. Additionally, we can one-hot encode the categorical values, though not the ordinal one that is tumour stage.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

X_genes = csr_matrix(scaled_data)
encoder = OneHotEncoder(handle_unknown="ignore")
encoded = encoder.fit_transform(metadata[["characteristics.tag.gender", "source.location"]])
X_combined = hstack([scaled_data, encoded])

<a name='03'></a>
# Part C. Model Evaluation

Evaluating the performance of a machine learning model goes beyond just looking at accuracy, as accuracy alone can be misleading, especially in cases where the dataset is imbalanced or where different types of errors have different consequences
- **Detecting Overfitting/Underfitting**: A learning curve can help you understand whether your model is overfitting (performing well on training data but poorly on validation data) or underfitting (performing poorly on both training and validation data)
- **ROC**: The ROC curve shows how well your model distinguishes between classes. It helps in selecting the optimal threshold for classification decisions, particularly when the cost of false positives and false negatives differs significantly. ROC curves are often used to compare models. 
- **Confusion matrix**: From the confusion matrix, you can derive other important metrics like precision, recall, F1-score, and specificity, which give a better understanding of how your model is performing across different classes


### <span style="background-color: lightyellow;">Evaluation Task</span>
- Review the documentation for `sklearn.metrics` and `learning_curve`.
- Select appropriate metrics for your dataset, including accuracy and indicators of overfitting or underfitting.
- Implement functions to compute and assess these metrics. It is allowed to use libraries.
- Add the evaluation functions as a Python module to your repository.
- Update the evaluation documentation to clearly explain your choices for the evaluation metrics and the rationale behind them.

See also: [Model evaluation video](https://video.hanze.nl/media/model-evaluation/0_gybpnhq7)


In [ ]:
# YOUR CODE HERE TO DEMONSTRATE THE USAGE OF THE EVALUATION MODULE
y.value_counts()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X_combined,y,test_size=0.2,random_state=8,stratify=y)

model = LogisticRegression(solver="saga",C=0.1)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(classification_report(y_test, predictions))

The report betrays something that was already quite evident, the classifier struggles a lot with how few instances some of the tumour types has. On those that it does have information on, it seems to be doing somewhat better. The precision and recall seem to be quite balanced. Preferably, we have a model that has a bit more of a tendency towards high precision because we'd rather someone be classified as having a certain tumour than missing out entirely. Better safe than sorry.


<a name='04'></a>
## Part D. Ensemble learning
Ensemble learning is a powerful machine learning technique that combines the predictions of multiple models to improve overall performance, robustness, and accuracy. Rather than relying on a single model, ensemble learning methods aggregate the results of several models—often called "weak learners"—to produce a stronger predictive model. The key idea behind ensemble learning is that by combining models, the weaknesses of individual models can be offset, leading to better generalization on unseen data. Popular ensemble ML algorithms are the `Random Forest` and `XGBoost`. 
Here’s an improved version of the introduction:

### voting algorithms
Voting algorithms in ensemble learning combine the predictions of multiple classifiers to make a final decision, typically based on the consensus or weighted agreement among models. Two primary types of voting are commonly used: hard voting, where the final prediction is determined by the majority vote, and soft voting, which uses the weighted average of predicted probabilities to determine the outcome. Several sites explain the hard and soft voting algorithm. A clear explanation can be found on https://www.baeldung.com/cs/hard-vs-soft-voting-classifiers


### <span style="background-color: lightyellow;">Ensemble Task</span>
- Review the [Study Case notebook on ensemble learning](..Study_Cases/study_case_bagging_boosting.ipynb).
- Try three different algorithms for classification of your data label.
- Implement a hard and soft voting algorithm for model aggregation.
- Compare the performance of the voting algorithm with that of a boosting or bagging algorithm.
- Update your repository with the voting algorithm class

See also: [ensemle learning video](https://video.hanze.nl/media/Ensemble/0_sue5v33g)


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn import tree

model = LogisticRegression(solver="saga",C=0.1)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

gnb = GaussianNB()
gnb.fit(X_train.toarray(), y_train)
y_pred = gnb.predict(X_test.toarray())

clf = tree.DecisionTreeClassifier(criterion="entropy", max_depth=20)
clf.fit(X_train, y_train)
predicted_sklearn = clf.predict(X_test)
predictions = np.array([y_pred, predictions, predicted_sklearn])



In [ ]:
# YOUR CODE HERE WITH THE VOTING ALGORITHMS


def vote_soft(probabilities):
    avg_probabilities = np.mean(probabilities, axis=0)
    return np.argmax(avg_probabilities, axis=1)

def vote_hard(predictions):
    return np.median(predictions, axis=0)

In [ ]:
# YOUR CODE HERE TO COMPARE VOTING OUTCOME WITH A BAGGING OR BOOSTING ALGORITHM FROM SKLEARN
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[('model1', model),('model2', gnb),
        ('model3', clf)],voting='soft')

voting.fit(X_train.toarray(), y_train)
predictions = voting.predict(X_test.toarray())

print(classification_report(y_test, predictions))

<a name='05'></a>
## Part E. Pipelines
The sklearn pipeline function is a tool in machine learning that simplifies the workflow by encapsulating all the steps involved in a single object. It offers advantages such as simplicity, reproducibility, efficiency, flexibility, and integration. The Pipeline is built using a list of (key, value) pairs, where the key is a string containing the name you want to give this step and value is an estimator object (the method to be executed).

### <span style="background-color: lightyellow;">Pipeline Task</span>
- Read the [Study Case notebook for a pipeline functions](..Study_Cases/study_case_pipeline.ipynb) to understand the principle of the pipeline function
- Implement a pipeline which prepares and classifies data 
- Use a `GridSearchCV` object with the `Pipeline` object and a parameter grid to optimize choose the best hyper parameters

In [ ]:
# YOUR CODE HERE WITH THE PIPELINE IMPLEMENTATION
from sklearn.model_selection import GridSearchCV
param_grid = {"C": np.logspace(-3, 3, 7)}  # [0.001, 0.01, 0.1, 1, 10, 100, 1000]

model = LogisticRegression()
grid_search = GridSearchCV(model, param_grid, cv=5, scoring="accuracy")
grid_search.fit(X_train, y_train)

print("Best C:", grid_search.best_params_["C"])

## <span style="background-color: lightyellow;">Bring it all together</span>

By now, you've developed code snippets for model evaluation, optimization, and data improvement. Now, leverage these skills to build a classification model using the Clinical and Genetic Lung data. Make sure that you log your experiments. 

Once you're satisfied with the model, upload the relevant code to your repository and or refactor code with new insights. Furthermore, take a moment to reflect on its applicability and potential real-world impact. Update your evaluation document(s) with these findings in your repository. 

## Bonus: ML for operations

If we intend to deploy the model in a real-world application, it's more efficient to save and reuse the trained model rather than retraining it each time. 

- Read the blog: https://neptune.ai/blog/saving-trained-model-in-python
- Write three python files 
    1) a train_model python file 
    2) a use_model python file
    3) a retrain_model python file that adds new data to the original training data and updates the model
- Update your repository
